<a href="https://colab.research.google.com/github/vrundakamboj/codealpha_task_1/blob/main/Speech_Recognition_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install librosa soundfile tensorflow scikit-learn matplotlib -q

In [ ]:
!pip install gdown -q
import zipfile, os

file_id = '1z3z1aEaUYTn2KljP1j7u68p-4tRYimG9'
!gdown --id {file_id} -O /content/RAVDESS.zip

extract_path = '/content/RAVDESS'
os.makedirs(extract_path, exist_ok=True)
with zipfile.ZipFile('/content/RAVDESS.zip', 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"✅ Extracted RAVDESS to {extract_path}")

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1z3z1aEaUYTn2KljP1j7u68p-4tRYimG9
From (redirected): https://drive.google.com/uc?id=1z3z1aEaUYTn2KljP1j7u68p-4tRYimG9&confirm=t&uuid=48bf263c-e611-4611-b8a0-2e81de8f2e97
To: /content/RAVDESS.zip
100% 450M/450M [00:11<00:00, 38.8MB/s]
✅ Extracted RAVDESS to /content/RAVDESS


In [ ]:
import glob, os

emotion_map = {
    '01': 'neutral', '02': 'calm', '03': 'happy', '04': 'sad',
    '05': 'angry', '06': 'fearful', '07': 'disgust', '08': 'surprised'
}

audio_files = glob.glob('/content/RAVDESS/**/*.wav', recursive=True)
print(f"Found {len(audio_files)} audio files (before dedup)")

seen = set()
data = []
for f in audio_files:
    fname = os.path.basename(f)
    if fname in seen:
        continue
    seen.add(fname)
    parts = fname.split('-')
    if len(parts) >= 3:
        emotion_code = parts[2]
        if emotion_code in emotion_map:
            data.append({'path': f, 'emotion': emotion_map[emotion_code]})

print(f"Labeled {len(data)} files (after dedup)")

# Merge 'calm' into 'neutral' — they're acoustically very similar and often confused
for item in data:
    if item['emotion'] == 'calm':
        item['emotion'] = 'neutral'

print(f"Final classes: {set(d['emotion'] for d in data)}")
print(f"Class distribution: {[(e, sum(1 for d in data if d['emotion']==e)) for e in set(d['emotion'] for d in data)]}")

Found 2880 audio files (before dedup)
Labeled 1440 files (after dedup)
Final classes: {'fearful', 'disgust', 'angry', 'surprised', 'neutral', 'sad', 'happy'}
Class distribution: [('fearful', 192), ('disgust', 192), ('angry', 192), ('surprised', 192), ('neutral', 288), ('sad', 192), ('happy', 192)]


In [ ]:
import librosa
import numpy as np
from tqdm import tqdm

def get_mfcc(audio, sr, max_len=174, n_mfcc=40):
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
    if mfcc.shape[1] < max_len:
        mfcc = np.pad(mfcc, ((0,0),(0, max_len - mfcc.shape[1])), mode='constant')
    else:
        mfcc = mfcc[:, :max_len]
    return mfcc

X, y = [], []
for item in tqdm(data):
    try:
        audio, sr = librosa.load(item['path'], sr=22050)

        # Original
        X.append(get_mfcc(audio, sr)); y.append(item['emotion'])

        # Augmented: add noise
        noisy = audio + np.random.normal(0, 0.005, audio.shape)
        X.append(get_mfcc(noisy, sr)); y.append(item['emotion'])

        # Augmented: pitch shift
        pitched = librosa.effects.pitch_shift(audio, sr=sr, n_steps=2)
        X.append(get_mfcc(pitched, sr)); y.append(item['emotion'])

        # Augmented: time stretch
        stretched = librosa.effects.time_stretch(audio, rate=0.9)
        X.append(get_mfcc(stretched, sr)); y.append(item['emotion'])

    except Exception as e:
        print(f"Skipped {item['path']}: {e}")

X = np.array(X)
y = np.array(y)
print(X.shape, y.shape)  # should be ~4x original: ~5760 samples

100%|██████████| 1440/1440 [05:16<00:00,  4.55it/s]


(5760, 40, 174) (5760,)


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_onehot = to_categorical(y_encoded)

X = X[..., np.newaxis]  # add channel dim for CNN: (samples, n_mfcc, time, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_onehot, test_size=0.2, random_state=42, stratify=y_encoded
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Classes:", le.classes_)

Train: (4608, 40, 174, 1) Test: (1152, 40, 174, 1)
Classes: ['angry' 'disgust' 'fearful' 'happy' 'neutral' 'sad' 'surprised']


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, BatchNormalization,
                                       Dropout, Flatten, Dense, Reshape, LSTM)

n_classes = y_onehot.shape[1]
input_shape = X_train.shape[1:]  # (n_mfcc, time, 1)

model = Sequential([
    Conv2D(32, (3,3), activation='relu', padding='same', input_shape=input_shape),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.3),

    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.3),

    Conv2D(128, (3,3), activation='relu', padding='same'),   # NEW: 3rd conv block
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.3),

    Reshape((-1, 128)),   # NEW: must match the 128 filters above (was 64 before)
    LSTM(128, return_sequences=False),
    Dropout(0.4),

    Dense(64, activation='relu'),
    Dense(n_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 40, 174, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 40, 174, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 20, 87, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 20, 87, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 20, 87, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 20, 87, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 43, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 10, 43, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 10, 43, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 10, 43, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 5, 21, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 5, 21, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 105, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           455 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 233,863 (913.53 KB)

 Trainable params: 233,415 (911.78 KB)

 Non-trainable params: 448 (1.75 KB)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.5, patience=5)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32,
    callbacks=callbacks
)

Epoch 1/50
144/144 ━━━━━━━━━━━━━━━━━━━━ 141s 935ms/step - accuracy: 0.2786 - loss: 1.8020 - val_accuracy: 0.3194 - val_loss: 1.7441 - learning_rate: 0.0010
Epoch 2/50
144/144 ━━━━━━━━━━━━━━━━━━━━ 140s 927ms/step - accuracy: 0.3637 - loss: 1.6383 - val_accuracy: 0.3785 - val_loss: 1.6048 - learning_rate: 0.0010
Epoch 3/50
144/144 ━━━━━━━━━━━━━━━━━━━━ 147s 957ms/step - accuracy: 0.4169 - loss: 1.5284 - val_accuracy: 0.4306 - val_loss: 1.5140 - learning_rate: 0.0010
Epoch 4/50
144/144 ━━━━━━━━━━━━━━━━━━━━ 137s 928ms/step - accuracy: 0.4512 - loss: 1.4529 - val_accuracy: 0.4661 - val_loss: 1.4038 - learning_rate: 0.0010
Epoch 5/50
144/144 ━━━━━━━━━━━━━━━━━━━━ 137s 896ms/step - accuracy: 0.4865 - loss: 1.3685 - val_accuracy: 0.5095 - val_loss: 1.3305 - learning_rate: 0.0010
Epoch 6/50
144/144 ━━━━━━━━━━━━━━━━━━━━ 126s 879ms/step - accuracy: 0.5043 - loss: 1.3144 - val_accuracy: 0.5122 - val_loss: 1.3342 - learning_rate: 0.0010
Epoch 7/50
144/144 ━━━━━━━━━━━━━━━━━━━━ 145s 901ms/step - accura

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

loss, acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {acc*100:.2f}%")

y_pred = model.predict(X_test).argmax(axis=1)
y_true = y_test.argmax(axis=1)

print(classification_report(y_true, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_, cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix')
plt.show()

plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.legend(); plt.title('Accuracy over epochs'); plt.show()

In [ ]:
from google.colab import files
import pickle

model.save('/content/speech_emotion_model.h5')
with open('/content/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

files.download('/content/speech_emotion_model.h5')
files.download('/content/label_encoder.pkl')